# 16.5 Padding 花掉多少工作？

# 第 16 章：由量測帶出效率 BKM

前置：已有第4章能forward與生成的模型。像整理廚房，先找哪一步最慢再改工作流程。CPU小例只驗證數值與資料契約；GPU速度、峯值記憶體與編譯收益必須在讀者的硬體實測。




In [ ]:
from pathlib import Path
import os
import subprocess
import sys

# Colab 會先下載專案；本機 Jupyter 沿用已開啟的 repo。
try:
    import google.colab
except ImportError:
    in_colab = False
else:
    in_colab = True

if in_colab:
    root = Path("/content/tiny-perceptron-vlm")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/birdhackor/tiny-perceptron-vlm.git", str(root)],
            env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(root)], check=True)
    os.chdir(root)

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("本機請先依 course/first-steps.md 開啟 repo；免安裝練習可使用本節的 Colab 入口")
sys.path.insert(0, str(root))
import torch
from torch import nn
from torch.nn import functional as F

torch.set_num_threads(1)
torch.manual_seed(42)

**先看圖：兩份文件隔開：EOS 不等於牆**

例中0–1為文件A，2–3為文件B。

橘色邊框只提示閱讀順序，靜態標註一直保留。系統的減少動態效果設定會停用動畫。


In [ ]:
from IPython.display import SVG, display

display(SVG(filename=str(root / "course/figures/packing.svg")))

**先想一想：**加EOS就能阻止後面文件看前文嗎？

packing把短文件放同一長序列，減少PAD。若希望文件互相隔離，要另外設segment mask；EOS只是結束符，不是看不見的牆。

**把直覺寫成數學：**allowed=causal AND same_segment；也要處理每文件position與boundary targets。

**最小實驗：**以下程式只處理這節的問題。


In [ ]:
from tiny_perceptron.attention import attention_mask

position = torch.arange(5)
segment = torch.tensor([[0, 0, 1, 1, 1]])
allowed = attention_mask(position, position, segments=segment)
print(allowed[0, 0].int())
assert not allowed[0, 0, 4, 0] and allowed[0, 0, 4, 2]

**如何讀結果：**此圖是隔離契約；如果刻意串接文件，要明確允許跨文件context。兩種不能混着比較。

**只改一件事：**只移除segments，觀察跨文件可見位置。
